# Reflection

- 【执行 -> 反思 -> 优化】迭代：
    - Execution：执行。agent使用ReAct或Plan-and-Solve尝试完成任务，生成一个初步的解决方案或行动轨迹，为“初稿”。
    - Reflection：反思。调用一个独立、带特殊提示词的LLM来扮演“评审”，从多维度评估“初稿”，如事实性错误、逻辑漏洞、效率问题、遗漏信息。根据评估，生成一段结构化的“反馈”，指出问题和改进建议。
    - Refinement：优化。agent将初稿和反馈作为新的上下文，再次调用LLM，生成一个更完善的“修订稿”。

---

- 抽象工作流程：假设$O_i$是第 i 次迭代产生的输出（$O_0$是初始输出），反思模型 $\pi_{\text{reflect}}$ 会生成针对 $O_i$ 的反馈 $F_i$：
$$F_i = \pi_{\text{reflect}}(\text{Task}, O_i)$$
随后，优化模型 $\pi_{\text{refine}}$ 会结合原始任务、上一版输出以及反馈，生成新一版的输出 $O_{i+1}$：
$$ O_{i+1} = \pi_{\text{refine}}(\text{Task}, O_i, F_i) $$

---

- 与之前范式相比：
    - 提供了内部纠错回路，使其不再完全依赖外部工具的反馈（ReAct的Observation）；
    - 一次性的任务执行变成了一个持续优化的过程；
    - 构建了临时的 **“短期记忆”**，在**记忆管理机制**中记录经验。
- 分析：
    - 成本增加：模型调用开销（没进行一轮迭代至少要调用两次LLM）；任务延迟（串行过程）；提示词优化依赖度高
    - 核心收益：解决方案质量提高；鲁棒性和可靠性增强

In [1]:
import sys
sys.path.append(r"C:\Users\61075\PycharmProjects\learning\hello_agent")
from _LLM import AgentsLLM
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

load_dotenv(r"C:\Users\61075\PycharmProjects\learning\hello_agent\.env")

True

In [2]:
### 记忆模块

class Memory:
    """短期记忆模块，用于存储agent的行动和反思轨迹"""
    def __init__(self):
        """初始化一个空列表，用于存储记录"""
        self.records: List[Dict[str, Any]] = []

    def add_record(self, record_type: str, content: str):
        """
        向记忆中添加一条新纪录
        :param record_type: 记录的类型，"execution" or "reflection"
        :param content: 记录的具体内容，如生成的代码或反思的反馈
        """
        record = {"type": record_type, "content": content}
        self.records.append(record)
        print(f"%%% 记忆已更新，新增一条 {record_type} 记录")

    def get_trajectory(self) -> str:
        """将所有记忆记录格式化为一个连贯的字符串文本，用于构建prompt"""
        trajectory_parts = []
        for record in self.records:
            if record["type"] == "execution":
                trajectory_parts.append(f"--- 上一轮尝试（代码） ---\n{record['content']}")
            elif record["type"] == "reflection":
                trajectory_parts.append(f"--- 评审员反馈 ---\n{record['content']}")

        return "\n\n".join(trajectory_parts)

    def get_last_execution(self) -> Optional[str]:
        """获取最近一次的执行结果，如最新生成的代码
        如果不存在，则返回None"""
        for record in reversed(self.records):
            if record["type"] == "execution":
                return record["content"]
        return None

In [3]:
# prompt template - reflection

# Execution prompt: 初始执行提示词，首次尝试解决问题，内容直接
INITIAL_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。请根据以下要求，编写一个Python函数。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。

要求: {task}

请直接输出代码，不要包含任何额外的解释。
"""

# Reflection prompt: 反思提示词，对上一轮的代码进行批判性分析，并提供具体、可操作的反馈
REFLECT_PROMPT_TEMPLATE = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码的性能有极致的要求。
你的任务是审查以下Python代码，并专注于找出其在<strong>算法效率</strong>上的主要瓶颈。

# 原始任务:
{task}

# 待审查的代码:
```python
{code}
```

请分析该代码的时间复杂度，并思考是否存在一种<strong>算法上更优</strong>的解决方案来显著提升性能。
如果存在，请清晰地指出当前算法的不足，并提出具体的、可行的改进算法建议（例如，使用筛法替代试除法）。
如果代码在算法层面已经达到最优，才能回答“无需改进”。

请直接输出你的反馈，不要包含任何额外的解释。
"""

# Refinement prompt: 优化提示词，收到反馈后对进行修改
REFINE_PROMPT_TEMPLATE ="""
你是一位资深的Python程序员。你正在根据一位代码评审专家的反馈来优化你的代码。

# 原始任务:
{task}

# 你上一轮尝试的代码:
{last_code_attempt}
评审员的反馈：
{feedback}

请根据评审员的反馈，生成一个优化后的新版本代码。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。
请直接输出优化后的代码，不要包含任何额外的解释。
"""

In [8]:
class ReflectionAgent:
    def __init__(self, llm_client, max_iterations=3):
        self.llm_client = llm_client
        self.max_iterations = max_iterations
        self.memory = Memory()

    def _get_llm_response(self, prompt: str) -> str:
        """一个辅助方法，用于调用LLM并获取完整的流式响应。"""
        messages = [{"role": "user", "content": prompt}]
        response_text = self.llm_client.think(messages=messages) or ""
        return response_text

    def run(self, task: str):
        print(f"\n--- 开始处理任务 ---\n任务: {task}")

        # 1. 初始执行
        print("\n--- 正在进行初始尝试 ---")
        initial_prompt = INITIAL_PROMPT_TEMPLATE.format(task=task)
        initial_code = self._get_llm_response(initial_prompt)
        self.memory.add_record("execution", initial_code)

        # 2. 迭代循环：反思与优化
        for i in range(self.max_iterations):
            print(f"\n--- 第 {i+1}/{self.max_iterations} 次迭代 ---")

            # 2.1 反思
            print("\n-> 正在进行反思...")
            last_code = self.memory.get_last_execution()
            reflect_prompt = REFLECT_PROMPT_TEMPLATE.format(
                task=task,
                code=last_code
            )
            feedback = self._get_llm_response(reflect_prompt)
            self.memory.add_record("reflection", feedback)

            # 2.2 检查是否需要停止
            if "无需改进" in feedback:
                print("\n%%% 反思认为代码已无需改进，任务完成")
                break

            # 2.3 优化
            print("\n-> 正在进行优化...")
            refine_prompt = REFINE_PROMPT_TEMPLATE.format(
                task=task,
                last_code_attempt=last_code,
                feedback=feedback
            )
            refine_code = self._get_llm_response(refine_prompt)
            self.memory.add_record("execution", refine_code)

        final_code = self.memory.get_last_execution()
        print(f"\n--- 任务完成 ---\n最终生成的代码：\n```python\n{final_code}\n```")
        return final_code

In [9]:
llm_client = AgentsLLM()

agent = ReflectionAgent(llm_client, max_iterations=3)
agent.run(task="编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。")


--- 开始处理任务 ---
任务: 编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。

--- 正在进行初始尝试 ---
%%% 正在调用 llama-3.3-70b-versatile 模型...
%%% 大语言模型响应成功：
```python
def find_primes(n: int) -> list:
    """
    Finds all prime numbers between 1 and n (inclusive).

    Args:
        n (int): The upper limit for finding prime numbers.

    Returns:
        list: A list of prime numbers between 1 and n.

    Raises:
        ValueError: If n is less than 1.
    """
    if n < 1:
        raise ValueError("n must be greater than or equal to 1")

    primes = []
    for possible_prime in range(2, n + 1):
        is_prime = True
        for num in range(2, int(possible_prime ** 0.5) + 1):
            if possible_prime % num == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(possible_prime)
    return primes


def main():
    n = 100
    primes = find_primes(n)
    print(f"Prime numbers between 1 and {n}: {primes}")


if __name__ == "__main__":
    main()
```

'```python\ndef find_primes(n: int) -> list:\n    """\n    Finds all prime numbers between 1 and n (inclusive) using the Sieve of Eratosthenes algorithm.\n\n    Args:\n        n (int): The upper limit for finding prime numbers.\n\n    Returns:\n        list: A list of prime numbers between 1 and n.\n\n    Raises:\n        ValueError: If n is less than 1.\n    """\n    if n < 1:\n        raise ValueError("n must be greater than or equal to 1")\n\n    sieve = [True] * (n + 1)\n    sieve[0:2] = [False, False]  # 0 and 1 are not prime numbers\n    for num in range(2, int(n ** 0.5) + 1):\n        if sieve[num]:\n            for multiple in range(num * num, n + 1, num):\n                sieve[multiple] = False\n    return [num for num, is_prime in enumerate(sieve) if is_prime]\n\n\ndef main():\n    n = 100\n    primes = find_primes(n)\n    print(f"Prime numbers between 1 and {n}: {primes}")\n\n\nif __name__ == "__main__":\n    main()\n```'